In [1]:
import os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = Path("/kaggle/data")

print("Exists /kaggle/input:", INPUT_ROOT.exists())
print("Top-level /kaggle/input entries (first 50):")
if INPUT_ROOT.exists():
    print(sorted([p.name for p in INPUT_ROOT.iterdir()])[:50])

print("Exists /kaggle/data:", DATA_ROOT.exists())
print("Top-level /kaggle/data entries (first 50):")
if DATA_ROOT.exists():
    print(sorted([p.name for p in DATA_ROOT.iterdir()])[:50])



Exists /kaggle/input: True
Top-level /kaggle/input entries (first 50):
['description.md', 'jigsaw-toxic-comment-classification-challenge', 'sample_submission.csv', 'sample_submission.csv.zip', 'test.csv', 'test.csv.zip', 'train.csv', 'train.csv.zip']
Exists /kaggle/data: True
Top-level /kaggle/data entries (first 50):
['description.md', 'jigsaw-toxic-comment-classification-challenge', 'sample_submission.csv', 'sample_submission.csv.zip', 'test.csv', 'test.csv.zip', 'train.csv', 'train.csv.zip']


In [2]:
import numpy as np
import pandas as pd

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Change (score-moving): load train/test and train a simple text model instead of blending nonexistent external submissions.
# This keeps evaluation semantics (probabilities per label) and should raise AUC far above the 0.5 baseline.
train_path_candidates = [
    DATA_ROOT / "train.csv",
    DATA_ROOT / "jigsaw-toxic-comment-classification-challenge" / "train.csv",
    INPUT_ROOT / "train.csv",
    INPUT_ROOT / "jigsaw-toxic-comment-classification-challenge" / "train.csv",
]
test_path_candidates = [
    DATA_ROOT / "test.csv",
    DATA_ROOT / "jigsaw-toxic-comment-classification-challenge" / "test.csv",
    INPUT_ROOT / "test.csv",
    INPUT_ROOT / "jigsaw-toxic-comment-classification-challenge" / "test.csv",
]
sample_path_candidates = [
    DATA_ROOT / "sample_submission.csv",
    DATA_ROOT
    / "jigsaw-toxic-comment-classification-challenge"
    / "sample_submission.csv",
    INPUT_ROOT / "sample_submission.csv",
    INPUT_ROOT
    / "jigsaw-toxic-comment-classification-challenge"
    / "sample_submission.csv",
]

train_path = next((p for p in train_path_candidates if p.exists()), None)
test_path = next((p for p in test_path_candidates if p.exists()), None)
sample_path = next((p for p in sample_path_candidates if p.exists()), None)

if train_path is None or test_path is None or sample_path is None:
    raise FileNotFoundError(
        f"Could not locate required CSVs. train:{train_path} test:{test_path} sample:{sample_path}"
    )

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

expected_cols = ["id"] + label_cols
if list(sample_sub.columns) != expected_cols:
    missing = set(expected_cols) - set(sample_sub.columns)
    if missing:
        raise ValueError(f"sample_submission.csv missing required columns: {missing}")
    sample_sub = sample_sub[expected_cols]

# Basic validation
for c in ["id", "comment_text"] + label_cols:
    if c not in train_df.columns:
        raise ValueError(f"train.csv missing column: {c}")
for c in ["id", "comment_text"]:
    if c not in test_df.columns:
        raise ValueError(f"test.csv missing column: {c}")

print("Loaded:")
print(" - train:", train_path, "rows:", len(train_df))
print(" - test :", test_path, "rows:", len(test_df))
print(" - sample:", sample_path, "rows:", len(sample_sub))



Loaded:
 - train: /kaggle/data/train.csv rows: 159571
 - test : /kaggle/data/test.csv rows: 153164
 - sample: /kaggle/data/sample_submission.csv rows: 153164


In [3]:
# Change (score-moving): TF-IDF + OneVsRest Logistic Regression (classic strong baseline for this competition).
# Uses only sklearn (available in Kaggle even if not listed explicitly) and produces proper probabilities.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

X_train_text = train_df["comment_text"].fillna("").astype(str).values
Y_train = train_df[label_cols].astype(np.int8).values
X_test_text = test_df["comment_text"].fillna("").astype(str).values

# Keep runtime under control while maintaining strong performance.
# (Not an approximation like sampling; this is a standard vectorizer configuration.)
tfidf = TfidfVectorizer(
    strip_accents="unicode",
    lowercase=True,
    analyzer="word",
    token_pattern=r"(?u)\b\w\w+\b",
    ngram_range=(1, 2),
    max_features=200000,
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

base_lr = LogisticRegression(
    solver="liblinear",
    max_iter=200,
    C=4.0,
)

clf = OneVsRestClassifier(base_lr, n_jobs=1)
clf.fit(X_train, Y_train)

# Predict probabilities for each label
pred_test = clf.predict_proba(X_test).astype(np.float32)
pred_test = np.clip(pred_test, 0.0, 1.0)

print("Pred shape:", pred_test.shape)
print("Pred min/max:", float(pred_test.min()), float(pred_test.max()))



Pred shape: (153164, 6)
Pred min/max: 2.239625246147625e-05 1.0


In [4]:
# Change (correctness): Build submission by aligning to sample_submission ids and required column order.
# This ensures the output is always in the exact schema Kaggle expects.
pred_df = pd.DataFrame(pred_test, columns=label_cols)
pred_df.insert(0, "id", test_df["id"].values)

# Align with sample_sub ids (robust even if ordering differs)
sub = sample_sub[["id"]].merge(pred_df, on="id", how="left", validate="1:1")
if sub[label_cols].isna().any().any():
    # If any missing (shouldn't happen), fill safely with 0.5 to keep a valid submission.
    sub[label_cols] = sub[label_cols].fillna(0.5)

sub = sub[["id"] + label_cols]
sub[label_cols] = sub[label_cols].astype(np.float32).clip(0.0, 1.0)

print("Submission preview:")
print(sub.head())
print("Rows:", len(sub), "Cols:", list(sub.columns))
print("Any nulls:", sub.isna().any().any())



Submission preview:
                 id     toxic  severe_toxic   obscene    threat    insult  \
0  00001cee341fdb12  0.999767      0.218013  0.998847  0.062782  0.952196   
1  0000247867823ef7  0.003000      0.001067  0.001249  0.000323  0.004018   
2  00013b17ad220c46  0.012782      0.001966  0.007404  0.000752  0.006780   
3  00017563c3f7919a  0.002444      0.001361  0.001687  0.000745  0.001419   
4  00017695ad8997eb  0.013695      0.004173  0.004498  0.001075  0.004697   

   identity_hate  
0       0.412065  
1       0.001486  
2       0.001870  
3       0.000831  
4       0.001508  
Rows: 153164 Cols: ['id', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
Any nulls: False


In [5]:
out_path = Path("submission.csv")
sub.to_csv(out_path, index=False)

print("Wrote:", out_path.resolve())
print("Min/max per label:")
print(sub[label_cols].agg(["min", "max"]))

Wrote: /kaggle/working/submission.csv
Min/max per label:
        toxic  severe_toxic   obscene   threat    insult  identity_hate
min  0.000022      0.000030  0.000074  0.00003  0.000054       0.000036
max  1.000000      0.998957  1.000000  0.99846  1.000000       0.999993
